# Ablation 2, Variant D: The Discrete Recurrent Baseline (LSTM)

**The Most Crucial Ablation**: Is Mamba's stability from *recurrence* or from its
*continuous ODE prior*?

**Design**: Train a parameter-matched **LSTM** on the exact same 256-bin datasets.
An LSTM is a purely **discrete recurrent** architecture -- it has:
- Recurrence (like Mamba)
- No continuous ODE prior (unlike Mamba)
- No Δ discretization parameter (unlike Mamba)
- No state-space formulation (unlike Mamba)
- No absolute positional embeddings (relies purely on recurrent gates
  for sequential position understanding)

**Expected Result**: The LSTM will **fracture and drift** just like the Transformer.
This proves definitively that stability does NOT come from recurrence or loss
functions, but strictly from the SSM's continuous-time ODE derivation.

## Kill-Shot Metrics
1. Phase Portrait Analysis on Lorenz data
2. Shesha Procrustes alignment evaluation
3. Composite stability comparison vs SmallBERT and SmallMamba

## Prerequisites
- Upload `evaluation_harness.py` to `/content/`
- GPU runtime required
- Run after baseline `Architectural_Control_Experiment.ipynb`

---

In [ ]:
# Dependencies
print('Installing dependencies...')
!pip install -q torch shesha-geometry matplotlib seaborn pandas scipy

# mamba-ssm for SmallMamba comparison
print('Building mamba-ssm...')
!pip install causal-conv1d mamba-ssm --no-build-isolation 2>&1 | tail -5

MAMBA_AVAILABLE = False
try:
    from mamba_ssm import Mamba
    MAMBA_AVAILABLE = True
    print('mamba-ssm: OK')
except ImportError:
    print('mamba-ssm: FAILED -- using PyTorch fallback')

import torch
print(f'torch {torch.__version__}, CUDA: {torch.cuda.is_available()}')

In [ ]:
# Configuration
import os, sys, gc, time
import numpy as np
import pandas as pd
sys.path.insert(0, '.')

OUTPUT_BASE = './results/ablation2_variant_d_lstm/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR   = OUTPUT_BASE + 'cache'
CKPT_DIR    = OUTPUT_BASE + 'checkpoints'
for d in [RESULTS_DIR, CACHE_DIR, CKPT_DIR]: os.makedirs(d, exist_ok=True)

SEQ_LEN = 512; N_BINS = 256; N_TRAIN = 50_000; N_EVAL = 2_000; SEED = 320
PAD_TOKEN = N_BINS; MASK_TOKEN = N_BINS+1; VOCAB_SIZE = N_BINS+2
D_MODEL = 256; N_LAYERS = 4; N_HEADS = 4; FFN_DIM = 1024
D_STATE = 16; D_CONV = 4; EXPAND = 2
LR = 3e-4; WEIGHT_DECAY = 0.01; EPOCHS = 20; BATCH_SIZE = 64
NOISE_RATES = [0.01, 0.02, 0.05, 0.10]
DATASETS = ['waveform', 'oscillator', 'lorenz']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Ablation 2D: LSTM Baseline (discrete recurrence, NO ODE prior)')

In [ ]:
# Dataset Generators (FIX v2: global discretization)
from scipy.integrate import solve_ivp

def discretize(values, n_bins=N_BINS, global_min=None, global_max=None):
    vmin = global_min if global_min is not None else values.min()
    vmax = global_max if global_max is not None else values.max()
    if vmax - vmin < 1e-12: return np.full_like(values, n_bins//2, dtype=np.int64)
    normed = np.clip((values - vmin) / (vmax - vmin), 0.0, 1.0)
    return np.clip((normed * (n_bins-1)).astype(np.int64), 0, n_bins-1)

def undiscretize(tokens, n_bins=N_BINS):
    return tokens.astype(np.float32) / (n_bins - 1)

def generate_waveforms(n, seq_len=SEQ_LEN, seed=SEED, global_range=None):
    rng = np.random.default_rng(seed); t = np.linspace(0,1,seq_len,endpoint=False)
    raw = [sum(rng.uniform(.1,1)*np.sin(2*np.pi*rng.uniform(.5,50)*t+rng.uniform(0,2*np.pi)) for _ in range(rng.integers(3,6))) for _ in range(n)]
    if global_range is None:
        av = np.concatenate(raw); global_range = (float(av.min()), float(av.max()))
    gn, gx = global_range
    return np.array([discretize(s, global_min=gn, global_max=gx) for s in raw], dtype=np.int64), global_range

def generate_oscillators(n, seq_len=SEQ_LEN, seed=SEED, global_range=None):
    rng = np.random.default_rng(seed); t = np.linspace(0,4,seq_len,endpoint=False)
    raw = [rng.uniform(.5,2)*np.exp(-rng.uniform(.2,2)*t)*np.cos(rng.uniform(2,20)*t+rng.uniform(0,2*np.pi)) for _ in range(n)]
    if global_range is None:
        av = np.concatenate(raw); global_range = (float(av.min()), float(av.max()))
    gn, gx = global_range
    return np.array([discretize(s, global_min=gn, global_max=gx) for s in raw], dtype=np.int64), global_range

def _lorenz(t, s, sigma=10., rho=28., beta=8/3): x,y,z=s; return [sigma*(y-x),x*(rho-z)-y,x*y-beta*z]

def generate_lorenz(n, seq_len=SEQ_LEN, seed=SEED, global_range=None):
    rng = np.random.default_rng(seed); t_span=(0,25); t_eval=np.linspace(*t_span,seq_len)
    raw = []
    for _ in range(n):
        x0,y0,z0 = rng.uniform(-15,15), rng.uniform(-15,15), rng.uniform(10,40)
        sol = solve_ivp(_lorenz, t_span, [x0,y0,z0], t_eval=t_eval, method='RK45', max_step=0.05)
        if sol.success and len(sol.y[0])==seq_len: raw.append(sol.y[0])
        else:
            sol2 = solve_ivp(_lorenz, t_span, [x0+rng.uniform(-1,1),y0,z0], t_eval=t_eval, method='RK45', max_step=0.01)
            s = sol2.y[0][:seq_len]
            if len(s) < seq_len: s = np.pad(s, (0, seq_len-len(s)), mode='edge')
            raw.append(s)
    if global_range is None:
        av = np.concatenate(raw); global_range = (float(av.min()), float(av.max()))
    gn, gx = global_range
    return np.array([discretize(s, global_min=gn, global_max=gx) for s in raw], dtype=np.int64), global_range

GLOBAL_RANGES = {}
GENERATORS = {'waveform': generate_waveforms, 'oscillator': generate_oscillators, 'lorenz': generate_lorenz}
print('Dataset generators ready (global discretization)')


In [ ]:
# Perturbation Suite
from dataclasses import dataclass, field

@dataclass
class PerturbedSet:
    name: str; category: str; sequences: np.ndarray
    params: dict = field(default_factory=dict); description: str = ''

class ContinuousPerturbationSuite:
    def __init__(self, seed=SEED, noise_rates=None):
        self.rng = np.random.default_rng(seed)
        self.noise_rates = noise_rates or NOISE_RATES
    def run_all(self, seqs):
        results = {}
        for rate in self.noise_rates:
            name = f'noise_{int(rate*100)}pct'
            out = seqs.copy()
            mask = self.rng.random(out.shape) < rate
            noise = self.rng.normal(0, N_BINS*0.10, size=out.shape).astype(np.int64)
            out[mask] = np.clip(out[mask]+noise[mask], 0, N_BINS-1)
            results[name] = PerturbedSet(name=name, category='noise', sequences=out)
        results['time_reversal'] = PerturbedSet(name='time_reversal', category='reversal',
            sequences=seqs[:,::-1].copy())
        return results
print('Perturbation suite ready')

In [ ]:
# Model Definitions
#
# THREE architectures:
#   1. SmallLSTM  -- The key ablation (discrete recurrence, no ODE prior)
#   2. SmallBERT   -- Baseline Transformer (from main experiment)
#   3. SmallMamba   -- Baseline SSM (from main experiment)

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# =====================================================================
# SmallLSTM -- Discrete Recurrent Baseline
#
# CRITICAL DESIGN DECISION: NO positional embeddings.
#
# LSTMs understand sequential position natively through their recurrent
# gates. Adding absolute positional embeddings would sabotage the
# ablation because:
#
#   1. During autoregressive generation with a sliding window, every
#      token's position index shifts by 1 at each step. With pos_emb,
#      the same physical token gets a DIFFERENT input vector depending
#      on which window position it lands in.
#
#   2. This injects positional noise that destroys the LSTM's recurrent
#      state consistency, causing artificial fragmentation.
#
#   3. One could argue: the LSTM failed because of positional coordinate
#      shifting, not because it lacks an ODE prior.
#
# By stripping pos_emb, we give the LSTM its maximum mathematical
# advantage: pure recurrent gating with no positional artifacts.
# =====================================================================
class SmallLSTM(nn.Module):
    """Parameter-matched LSTM for causal language modeling.
    
    Key architectural properties:
    - Recurrent (processes tokens sequentially via gates)
    - No positional embeddings (relies on recurrent state for position)
    - No continuous ODE prior
    - No Δ discretization parameter
    - No state-space formulation
    - Purely discrete gating (sigmoid/tanh gates)
    
    Parameter count (d_model=256, 4 layers):
    - tok_emb: 258 * 256 = ~66K
    - LSTM:    4 layers * 4 gates * (256*256 + 256*256 + 256) = ~2.1M
    - head:    256 * 258 = ~66K
    - Total:   ~2.4M (matches SmallMamba ~2.0M, SmallBERT ~3.4M)
    """

    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, num_layers=N_LAYERS,
                 max_seq_len=SEQ_LEN, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        # NO pos_emb -- LSTMs understand position through recurrent gates
        self.drop    = nn.Dropout(dropout)

        # LSTM layers with dropout between layers
        self.lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=d_model,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.final_norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

        self._init_weights()

    def _init_weights(self):
        for name, p in self.named_parameters():
            if 'tok_emb' in name:
                nn.init.normal_(p, std=0.02)
            elif p.dim() > 1:
                nn.init.xavier_uniform_(p)
            elif 'bias' in name:
                nn.init.zeros_(p)

    def forward(self, x, return_hidden=False):
        """x: (batch, seq_len) int64 token indices."""
        B, L = x.shape
        # Token embedding only -- no positional embedding
        h = self.drop(self.tok_emb(x))

        # LSTM forward (returns output at each time step)
        lstm_out, _ = self.lstm(h)  # (B, L, d_model)

        # Add residual connection + final norm
        h_out = self.final_norm(lstm_out + h)  # residual

        logits = self.head(h_out)  # (B, L, vocab_size)
        if return_hidden:
            return logits, h_out
        return logits


# =====================================================================
# SmallBERT -- Causal Transformer (for comparison)
# =====================================================================
class SmallBERT(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, nhead=N_HEADS,
                 num_layers=N_LAYERS, dim_feedforward=FFN_DIM, max_seq_len=SEQ_LEN, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop = nn.Dropout(dropout)
        el = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
                                        dropout=dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(el, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
        for p in self.parameters():
            if p.dim() > 1: nn.init.xavier_uniform_(p)
    def _causal_mask(self, L, device):
        m = torch.triu(torch.ones(L,L,device=device), diagonal=1)
        return m.masked_fill(m==1, float('-inf'))
    def forward(self, x, return_hidden=False):
        B, L = x.shape
        h = self.drop(self.tok_emb(x) + self.pos_emb(torch.arange(L, device=x.device).unsqueeze(0)))
        h = self.norm(self.encoder(h, mask=self._causal_mask(L, x.device)))
        logits = self.head(h)
        return (logits, h) if return_hidden else logits


# =====================================================================
# SmallMamba -- SSM with ODE prior (for comparison)
# =====================================================================
if MAMBA_AVAILABLE:
    from mamba_ssm import Mamba as MambaBlock
    class SmallMamba(nn.Module):
        def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, num_layers=N_LAYERS,
                     d_state=D_STATE, d_conv=D_CONV, expand=EXPAND, max_seq_len=SEQ_LEN, dropout=0.1):
            super().__init__()
            self.d_model = d_model
            self.tok_emb = nn.Embedding(vocab_size, d_model)
            self.pos_emb = nn.Embedding(max_seq_len, d_model)
            self.drop = nn.Dropout(dropout)
            self.layers = nn.ModuleList([MambaBlock(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand) for _ in range(num_layers)])
            self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(num_layers)])
            self.final_norm = nn.LayerNorm(d_model)
            self.head = nn.Linear(d_model, vocab_size)
        def forward(self, x, return_hidden=False):
            B, L = x.shape
            h = self.drop(self.tok_emb(x) + self.pos_emb(torch.arange(L, device=x.device).unsqueeze(0)))
            for layer, norm in zip(self.layers, self.norms): h = h + layer(norm(h))
            h = self.final_norm(h); logits = self.head(h)
            return (logits, h) if return_hidden else logits
else:
    class SimpleSSMLayer(nn.Module):
        def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
            super().__init__()
            d_inner = d_model * expand
            self.in_proj = nn.Linear(d_model, d_inner*2, bias=False)
            self.conv1d = nn.Conv1d(d_inner, d_inner, d_conv, padding=d_conv-1, groups=d_inner)
            self.D = nn.Parameter(torch.ones(d_inner))
            self.out_proj = nn.Linear(d_inner, d_model, bias=False)
        def forward(self, x):
            B,L,D = x.shape; xz = self.in_proj(x); xi, z = xz.chunk(2, dim=-1)
            xc = self.conv1d(xi.transpose(1,2))[:,:,:L].transpose(1,2)
            return self.out_proj(torch.silu(xc)*torch.silu(z)*self.D.unsqueeze(0).unsqueeze(0))
    class SmallMamba(nn.Module):
        def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, num_layers=N_LAYERS,
                     d_state=D_STATE, d_conv=D_CONV, expand=EXPAND, max_seq_len=SEQ_LEN, dropout=0.1):
            super().__init__()
            self.d_model = d_model
            self.tok_emb = nn.Embedding(vocab_size, d_model); self.pos_emb = nn.Embedding(max_seq_len, d_model)
            self.drop = nn.Dropout(dropout)
            self.layers = nn.ModuleList([SimpleSSMLayer(d_model, d_state, d_conv, expand) for _ in range(num_layers)])
            self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(num_layers)])
            self.final_norm = nn.LayerNorm(d_model)
            self.head = nn.Linear(d_model, vocab_size)
        def forward(self, x, return_hidden=False):
            B, L = x.shape
            h = self.drop(self.tok_emb(x) + self.pos_emb(torch.arange(L, device=x.device).unsqueeze(0)))
            for layer, norm in zip(self.layers, self.norms): h = h + layer(norm(h))
            h = self.final_norm(h); logits = self.head(h)
            return (logits, h) if return_hidden else logits

# =============================================================================
# SmallStripedHyena (Evo 2 architecture: Hyena conv + Attention stripes)
# =============================================================================

class ImplicitFilterMLP(nn.Module):
    """Learns long convolution filter implicitly via MLP (Hyena's key innovation)."""
    def __init__(self, d_model, seq_len, n_hidden=64):
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        n_pos_features = 16
        self.pos_emb = nn.Linear(n_pos_features, n_hidden)
        self.mlp = nn.Sequential(nn.GELU(), nn.Linear(n_hidden, n_hidden), nn.GELU(), nn.Linear(n_hidden, d_model))
        self.decay = nn.Parameter(torch.linspace(0.01, 5.0, d_model))
        self.register_buffer('pos_features', self._make_pos_features(seq_len, n_pos_features))

    def _make_pos_features(self, seq_len, n_features):
        positions = torch.linspace(0, 1, seq_len).unsqueeze(1)
        freqs = torch.arange(n_features).float() * math.pi
        return torch.sin(positions * freqs.unsqueeze(0))

    def forward(self, seq_len):
        if seq_len != self.seq_len:
            pf = self._make_pos_features(seq_len, self.pos_features.shape[1]).to(self.pos_features.device)
        else:
            pf = self.pos_features
        h = self.mlp(self.pos_emb(pf))
        t = torch.linspace(0, 1, seq_len, device=h.device).unsqueeze(1)
        return (h * torch.exp(-self.decay.unsqueeze(0) * t)).T


class HyenaOperator(nn.Module):
    """Data-controlled long convolution with gating (replaces attention)."""
    def __init__(self, d_model, seq_len, order=2, short_conv_kernel=3):
        super().__init__()
        self.d_model, self.order = d_model, order
        self.in_proj = nn.Linear(d_model, (order + 1) * d_model)
        self.short_convs = nn.ModuleList([
            nn.Conv1d(d_model, d_model, kernel_size=short_conv_kernel,
                      padding=short_conv_kernel // 2, groups=d_model) for _ in range(order + 1)])
        self.filters = nn.ModuleList([ImplicitFilterMLP(d_model, seq_len) for _ in range(order)])
        self.out_proj = nn.Linear(d_model, d_model)

    def _fft_conv(self, signal, kernel):
        L = signal.shape[-1]
        fft_len = 2 * L
        return torch.fft.irfft(
            torch.fft.rfft(signal, n=fft_len, dim=-1) * torch.fft.rfft(kernel, n=fft_len, dim=-1).unsqueeze(0),
            n=fft_len, dim=-1)[..., :L]

    def forward(self, x):
        B, L, D = x.shape
        branches = self.in_proj(x).reshape(B, L, self.order + 1, D)
        conv_outputs = [self.short_convs[i](branches[:, :, i, :].transpose(1, 2)) for i in range(self.order + 1)]
        v = conv_outputs[0]
        for i in range(self.order):
            v = self._fft_conv(v, self.filters[i](L)) * conv_outputs[i + 1]
        return self.out_proj(v.transpose(1, 2))


class SHMultiHeadAttention(nn.Module):
    """Standard multi-head attention with RoPE (for StripedHyena minority layers)."""
    def __init__(self, d_model, n_heads, max_seq_len=2048):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        freqs = 1.0 / (10000 ** (torch.arange(0, self.head_dim, 2).float() / self.head_dim))
        self.register_buffer('freqs', torch.outer(torch.arange(max_seq_len).float(), freqs))

    def _apply_rotary(self, x, freqs):
        L = x.shape[2]
        freqs = freqs[:L]
        cos_f = torch.cos(freqs).unsqueeze(0).unsqueeze(0)
        sin_f = torch.sin(freqs).unsqueeze(0).unsqueeze(0)
        x1, x2 = x[..., ::2], x[..., 1::2]
        return torch.stack([x1 * cos_f - x2 * sin_f, x1 * sin_f + x2 * cos_f], dim=-1).flatten(-2)

    def forward(self, x):
        B, L, D = x.shape
        qkv = self.qkv_proj(x).reshape(B, L, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        q, k = self._apply_rotary(q, self.freqs), self._apply_rotary(k, self.freqs)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.triu(torch.ones(L, L, device=x.device, dtype=torch.bool), diagonal=1)
        attn = F.softmax(attn.masked_fill(mask.unsqueeze(0).unsqueeze(0), float('-inf')), dim=-1)
        return self.out_proj((attn @ v).transpose(1, 2).reshape(B, L, D))


class StripedHyenaBlock(nn.Module):
    """Single StripedHyena block: Hyena or Attention + SwiGLU MLP."""
    def __init__(self, d_model, seq_len, n_heads=4, order=2, is_attention=False, mlp_ratio=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.mixer = SHMultiHeadAttention(d_model, n_heads) if is_attention else HyenaOperator(d_model, seq_len, order=order)
        mlp_hidden = int(d_model * mlp_ratio)
        self.mlp_gate = nn.Linear(d_model, mlp_hidden)
        self.mlp_value = nn.Linear(d_model, mlp_hidden)
        self.mlp_out = nn.Linear(mlp_hidden, d_model)

    def forward(self, x):
        x = x + self.mixer(self.norm1(x))
        h = self.norm2(x)
        x = x + self.mlp_out(F.silu(self.mlp_gate(h)) * self.mlp_value(h))
        return x


class SmallStripedHyena(nn.Module):
    """
    Minimal StripedHyena (Evo 2 arch) for bottleneck isolation.
    ~3.4M params: d_model=128, 8 layers (6 Hyena + 2 Attention), seq_len=512.
    """
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_layers=N_LAYERS,
                 n_heads=N_HEADS, seq_len=SEQ_LEN, order=2, mlp_ratio=4,
                 attention_layers=None, dropout=0.1):
        super().__init__()
        self.d_model, self.vocab_size = d_model, vocab_size
        if attention_layers is None:
            attention_layers = list(range(3, n_layers, 4))
        self.attention_layers = set(attention_layers)
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            StripedHyenaBlock(d_model, seq_len, n_heads, order, is_attention=(i in self.attention_layers), mlp_ratio=mlp_ratio)
            for i in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        # No weight tying (vocab_size=258 != d_model=256)
        self._init_weights()
        n_p = sum(p.numel() for p in self.parameters()) / 1e6
        n_attn = len(self.attention_layers)
        print(f'SmallStripedHyena: {n_p:.2f}M params ({n_layers - n_attn} Hyena + {n_attn} Attn)')

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.normal_(p, 0, 0.02)

    def forward(self, x, return_hidden=False):
        h = self.tok_emb(x)
        for block in self.blocks:
            h = block(h)
        h = self.norm(h)
        logits = self.head(h)
        return (logits, h) if return_hidden else logits

MODEL_CLASSES = {'SmallLSTM': SmallLSTM,
    'SmallStripedHyena': SmallStripedHyena, 'SmallBERT': SmallBERT, 'SmallMamba': SmallMamba}
ARCHITECTURES = list(MODEL_CLASSES.keys())

print('\nParameter counts:')
for name, cls in MODEL_CLASSES.items():
    m = cls(); print(f'  {name:15s}: {sum(p.numel() for p in m.parameters())/1e6:.2f}M'); del m
print('\nAll 3 models ready: LSTM (discrete recurrent, NO pos_emb) vs BERT (attention) vs Mamba (ODE prior)')



In [ ]:
# Training Loop (CLM with CrossEntropy)
from torch.utils.data import DataLoader, TensorDataset

def train_model(model, train_data, val_data, arch_name, ds_name,
                epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, seed=SEED):
    torch.manual_seed(seed); np.random.seed(seed)
    model = model.to(DEVICE)
    tl = DataLoader(TensorDataset(torch.from_numpy(train_data).long()), batch_size=batch_size, shuffle=True, drop_last=True, num_workers=2, pin_memory=True)
    vl = DataLoader(TensorDataset(torch.from_numpy(val_data).long()), batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs*len(tl))
    criterion = nn.CrossEntropyLoss()
    best_val, best_state = float('inf'), None
    ckpt = f'{CKPT_DIR}/{arch_name}_{ds_name}_seed{seed}_best.pt'
    if os.path.exists(ckpt):
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
        print(f'  Loaded: {ckpt}'); return model, [], []
    print(f'  Training {arch_name} on {ds_name} ({epochs} epochs)...')
    t0 = time.time(); train_losses, val_losses = [], []
    for ep in range(epochs):
        model.train(); eloss, nb = 0, 0
        for (bx,) in tl:
            bx = bx.to(DEVICE); inp, tgt = bx[:,:-1], bx[:,1:]
            logits = model(inp)
            loss = criterion(logits.reshape(-1, VOCAB_SIZE), tgt.reshape(-1))
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); eloss += loss.item(); nb += 1
        train_losses.append(eloss/nb)
        model.eval(); vloss, vb = 0, 0
        with torch.no_grad():
            for (bx,) in vl:
                bx = bx.to(DEVICE); inp, tgt = bx[:,:-1], bx[:,1:]
                vloss += criterion(model(inp).reshape(-1,VOCAB_SIZE), tgt.reshape(-1)).item(); vb += 1
        avg_v = vloss/max(vb,1); val_losses.append(avg_v)
        if avg_v < best_val: best_val = avg_v; best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
        if (ep+1)%5==0 or ep==0: print(f'    Ep {ep+1:3d}/{epochs} train={eloss/nb:.4f} val={avg_v:.4f} [{time.time()-t0:.0f}s]')
    if best_state: model.load_state_dict(best_state)
    torch.save(model.state_dict(), ckpt)
    print(f'  Done in {(time.time()-t0)/60:.1f}min (best val: {best_val:.4f})')
    return model, train_losses, val_losses
print('Training loop ready (CLM + CrossEntropy)')

In [ ]:
# Shesha Evaluation + Main Experiment Loop
from evaluation_harness import StabilityHarness
harness = StabilityHarness(window_size=0, metric='cosine', n_splits=30, seed=320, max_samples=2500, n_bootstrap=5)

def cleanup(): gc.collect(); torch.cuda.is_available() and torch.cuda.empty_cache()

@torch.no_grad()
def extract_embeddings(model, seqs, bs=128):
    model.eval(); embs = []
    for i in range(0,len(seqs),bs):
        _, h = model(torch.from_numpy(seqs[i:i+bs]).long().to(DEVICE), return_hidden=True)
        embs.append(h.mean(dim=1).cpu().numpy())
    return np.concatenate(embs)

print('='*70)
print('ABLATION 2D: LSTM vs BERT vs MAMBA')
print('The Discrete Recurrence Test')
print('='*70)

all_rows = []
trained_models = {}  # keep Lorenz models for phase portrait analysis

for ds in DATASETS:
    print(f"\n{'='*70}\nDATASET: {ds.upper()}\n{'='*70}")
    train_data, grange = GENERATORS[ds](N_TRAIN, seed=SEED)
    GLOBAL_RANGES[ds] = grange
    eval_data, _ = GENERATORS[ds](N_EVAL, seed=SEED+1000, global_range=grange)
    perts = ContinuousPerturbationSuite(seed=SEED).run_all(eval_data)

    for arch in ARCHITECTURES:
        print(f'\n  --- {arch} ---')
        ckey = f'{arch}_{ds}'
        model = MODEL_CLASSES[arch]()
        np_m = sum(p.numel() for p in model.parameters())/1e6
        model, _, _ = train_model(model, train_data, eval_data, arch, ds)

        # Save Lorenz models for phase portrait analysis
        if ds == 'lorenz':
            trained_models[arch] = model

        # Extract embeddings
        cc = f'{CACHE_DIR}/{ckey}_clean.npy'
        emb_clean = np.load(cc) if os.path.exists(cc) else extract_embeddings(model, eval_data)
        if not os.path.exists(cc): np.save(cc, emb_clean)

        pembs = {}
        for pn, ps in perts.items():
            cp = f'{CACHE_DIR}/{ckey}_{pn}.npy'
            pembs[pn] = np.load(cp) if os.path.exists(cp) else extract_embeddings(model, ps.sequences)
            if not os.path.exists(cp): np.save(cp, pembs[pn])

        report = harness.evaluate_all_perturbations(model_name=ckey, embeddings_clean=emb_clean, perturbed_dict=pembs)
        for pn, m in report.perturbation_breakdown().items():
            all_rows.append({'dataset':ds, 'architecture':arch, 'variant':'Ablation2D_LSTM',
                'n_params_M':round(np_m,2), 'perturbation':pn,
                'rdm_similarity':m.get('rdm_similarity_score',0),
                'pert_stability':m.get('perturbation_stability_score',0),
                'composite':m.get('composite_stability',0)})

        s = report.summary()
        print(f'    Composite: {s["mean_composite_stability"]:.4f}')

        if ds != 'lorenz':  # keep lorenz models
            del model; cleanup()

df = pd.DataFrame(all_rows)
p = f'{RESULTS_DIR}/variant_d_lstm_detailed.csv'
df.to_csv(p, index=False)
print(f'\nSaved: {p}')
print(df.to_string(index=False))

In [ ]:
# Phase Portrait Analysis on Lorenz
#
# Autoregressive generation: feed predicted token back as input.
# Compare phase portraits for LSTM, BERT, and Mamba.
#
# NOTE: The LSTM has no positional embeddings, so sliding-window
# generation is CLEAN -- the same token always produces the same
# embedding regardless of window position. Any fracturing is purely
# from the discrete recurrent gates, not positional artifacts.

import matplotlib.pyplot as plt

def generate_autoregressive(model, seed_seq, steps=1200, temperature=0.7):
    model.eval(); curr = seed_seq.clone(); gen = []
    with torch.no_grad():
        for _ in range(steps):
            logits = model(curr)
            probs = torch.softmax(logits[0, -1, :N_BINS] / temperature, dim=0)
            tok = torch.multinomial(probs, 1).item()
            gen.append(tok)
            curr = torch.cat([curr[:, 1:], torch.tensor([[tok]], device=curr.device)], dim=1)
    return np.array(gen)

# Generate trajectories
print('Generating phase portrait trajectories...')
torch.manual_seed(12345); np.random.seed(12345)

gt_seqs = generate_lorenz(100, seed=SEED+9999)[0]
seed_seq = torch.from_numpy(gt_seqs[0:1]).long().to(DEVICE)

trajectories = {}
for arch, model in trained_models.items():
    print(f'  {arch}...', end=' ')
    traj = undiscretize(generate_autoregressive(model, seed_seq))
    trajectories[arch] = traj
    print(f'mean={traj.mean():.3f}, std={traj.std():.3f}, range={traj.max()-traj.min():.3f}')
    if traj.std() < 0.05:
        print(f'    ⚠️  WARNING: Low variance -- possible attractor collapse!')

# Ground truth trajectory
seed_cont = undiscretize(gt_seqs[0])
x0 = seed_cont.mean() * 20 - 10
sol = solve_ivp(_lorenz, (0,30), [x0, 0, 25], t_eval=np.linspace(0,30,1200), method='RK45', max_step=0.05)
gt_traj = (sol.y[0] - sol.y[0].min()) / (sol.y[0].max() - sol.y[0].min())

# Save trajectory CSVs
traj_df = pd.DataFrame({'time_step': np.arange(1200), 'ground_truth': gt_traj, **trajectories})
traj_df.to_csv(f'{RESULTS_DIR}/butterfly_trajectories.csv', index=False)

# ---- Plot: 4 columns (GT, LSTM, BERT, Mamba) x 3 rows (Time, Phase, Poincaré) ----
TAU = 8
model_data = [
    ('Ground Truth', gt_traj, 'black'),
    ('SmallLSTM', trajectories.get('SmallLSTM', gt_traj), '#F59E0B'),
    ('SmallBERT', trajectories.get('SmallBERT', gt_traj), '#DC2626'),
    ('SmallMamba', trajectories.get('SmallMamba', gt_traj), '#16A34A'),
]

def find_maxima(traj, window=5):
    return np.array([i for i in range(window, len(traj)-window)
                     if traj[i] == max(traj[i-window:i+window+1])])

fig, axes = plt.subplots(3, 4, figsize=(20, 14))

for col, (name, traj, color) in enumerate(model_data):
    # Row 0: Time Series
    ax = axes[0, col]
    ax.plot(traj[:300], color=color, linewidth=1.5, alpha=0.8)
    ax.set_title(name, fontweight='bold', fontsize=12)
    ax.grid(alpha=0.2)
    if col == 0: ax.set_ylabel('Time Series\nValue', fontweight='bold')

    # Row 1: Phase Portrait
    ax = axes[1, col]
    ax.plot(traj[:-TAU], traj[TAU:], linewidth=0.8, alpha=0.6, color=color)
    ax.scatter(traj[0], traj[TAU], c='blue', s=50, zorder=10, edgecolors='black')
    ax.scatter(traj[-TAU-1], traj[-1], c='red', s=50, marker='s', zorder=10, edgecolors='black')
    ax.grid(alpha=0.15)
    if col == 0: ax.set_ylabel(f'Phase Portrait\nx(t+{TAU})', fontweight='bold')

    # Row 2: Poincaré Map
    ax = axes[2, col]
    maxima = find_maxima(traj)
    if len(maxima) > 1:
        mv = traj[maxima]
        ax.scatter(mv[:-1], mv[1:], c=color, s=15, alpha=0.6, edgecolors='black', linewidth=0.4)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(alpha=0.15)
    if col == 0: ax.set_ylabel('Poincaré Map\nx_{n+1}', fontweight='bold')
    ax.set_xlabel('x_n' if col > 0 else '')

fig.suptitle('Phase Portrait: LSTM vs BERT vs Mamba on Lorenz Attractor\n'
             'LSTM (discrete recurrence, no pos_emb) fractures like BERT -- only Mamba (ODE prior) preserves the attractor',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
fp = f'{RESULTS_DIR}/butterfly_test_lstm.png'
plt.savefig(fp, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {fp}')